# Fretwork Debug — CAGED Voiced: Audio → MIDI Notes → ASCII Tab

Plug in any audio file from Google Drive. This notebook runs:
1. **Basic Pitch** — shows every raw MIDI note detected (so you can spot octave errors at the source)
2. **`caged_voiced` fret assignment** — the exact algorithm from `AudioToTabCAGED__2_`
3. **ASCII tab output**

**Only Cell 3 needs editing** — set `AUDIO_FILE` and optionally `KEY_LABEL`.

In [1]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
import sys, subprocess, pkgutil, zipimport
if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = zipimport.zipimporter
def run(cmd): subprocess.check_call(cmd)
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'wheel', 'setuptools==80.9.0'])
run([sys.executable, '-m', 'pip', 'install', '-q',
     'librosa>=0.10', 'soundfile', 'mir-eval', 'pretty_midi', 'resampy==0.4.2', 'onnxruntime'])
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'basic-pitch==0.4.0'])
print('Done.')

Done.


In [3]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except ModuleNotFoundError:
    print('Not in Colab — skipping Drive mount.')

Mounted at /content/drive
Google Drive mounted.


In [4]:
# ── Cell 3: CONFIG ✏️ ─────────────────────────────────────────────────────────

# Path to your audio file in Google Drive
AUDIO_FILE = '/content/drive/MyDrive/Capstone/Audio/isThisItIntro.mp3'

# Optional: set the key so the CAGED box anchors to the right position.
# Examples: 'D major', 'A minor', 'G major', None (auto-flat if unknown)
KEY_LABEL = None

# Basic Pitch settings (match AudioToTabCAGED__2_ defaults)
BASIC_PITCH_AMPLITUDE_THRESHOLD = 0.4
BASIC_PITCH_MIN_MIDI = 40
BASIC_PITCH_MAX_MIDI = 88
ONSET_THRESHOLD = 0.5

# Tab render settings
ASSUMED_TEMPO_BPM     = 120
SUBDIVISIONS_PER_BEAT = 2
BEATS_PER_MEASURE     = 4
MEASURES_PER_LINE     = 4
COL_WIDTH             = 4

In [5]:
# ── Cell 4: Constants, fretboard, key database ────────────────────────────────
from pathlib import Path
from itertools import product
from collections import defaultdict
import math, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

OPEN_STRING_MIDI   = [40, 45, 50, 55, 59, 64]
STRING_NAMES       = ['low_E', 'A', 'D', 'G', 'B', 'high_E']
MAX_FRET           = 24
ONSET_TOLERANCE_SECONDS = 0.035
COMFORTABLE_SPAN   = 5
MAX_REACHABLE_SPAN = 7
LARGE_JUMP_THRESHOLD = 5
MAX_GROUP_CANDIDATES = 25

PITCH_CLASS_NAMES_SHARP = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
NOTE_TO_PC = {name: i for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}
NOTE_TO_PC.update({'Db':1,'Eb':3,'Gb':6,'Ab':8,'Bb':10})
PC_TO_NOTE = {i: name for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}

def midi_to_note_name(midi):
    midi = int(round(midi))
    return f"{PITCH_CLASS_NAMES_SHARP[midi % 12]}{midi // 12 - 1}"

# Build fretboard lookup
fretboard_rows = []
for si, om in enumerate(OPEN_STRING_MIDI):
    for fret in range(MAX_FRET + 1):
        fretboard_rows.append({'string': si, 'string_name': STRING_NAMES[si],
                                'fret': fret, 'midi': om + fret, 'pitch_class': (om + fret) % 12})
fretboard_df = pd.DataFrame(fretboard_rows)
MIDI_TO_POSITIONS = defaultdict(list)
for row in fretboard_df.to_dict('records'):
    MIDI_TO_POSITIONS[int(row['midi'])].append({
        'string': int(row['string']), 'string_name': row['string_name'],
        'fret': int(row['fret']), 'midi': int(row['midi']), 'pitch_class': int(row['pitch_class'])})

def get_possible_positions(midi_note, max_fret=MAX_FRET):
    return [p for p in MIDI_TO_POSITIONS.get(int(round(midi_note)), []) if 0 <= p['fret'] <= max_fret]

# Key database
MAJOR_STEPS = [2,2,1,2,2,2,1]
MINOR_STEPS = [2,1,2,2,1,2,2]

def derive_scale(root_pc, mode='major'):
    steps = MAJOR_STEPS if mode == 'major' else MINOR_STEPS
    pcs = [root_pc]; cur = root_pc
    for step in steps[:-1]:
        cur = (cur + step) % 12; pcs.append(cur)
    return pcs

def build_key_database():
    rows = []
    for root_name, root_pc in NOTE_TO_PC.items():
        if 'b' in root_name: continue
        for mode in ['major', 'minor']:
            scale_pcs = derive_scale(root_pc, mode)
            rows.append({'key': f'{root_name} {mode}', 'root': root_name,
                         'root_pc': root_pc, 'mode': mode, 'scale_pcs': scale_pcs})
    return pd.DataFrame(rows)

key_db = build_key_database()

def get_key_info(key_label):
    if key_label is None: return None
    s = str(key_label).replace(':', ' ').strip()
    toks = s.split()
    if len(toks) == 1 and toks[0] in NOTE_TO_PC: s = f'{toks[0]} major'
    match = key_db[key_db['key'] == s]
    return match.iloc[0].to_dict() if len(match) else None

def parse_key(key_label):
    info = get_key_info(key_label)
    return None if info is None else {'root_pc': int(info['root_pc']), 'mode': info['mode'],
                                       'scale_pcs': set(info['scale_pcs'])}

print('Constants, fretboard, key DB ready.')

Constants, fretboard, key DB ready.


In [6]:
# ── Cell 5: Shared cost helpers ───────────────────────────────────────────────

def group_span(frets):
    fretted = [f for f in frets if f > 0]
    return max(fretted) - min(fretted) if len(fretted) >= 2 else 0

def estimate_hand_position_from_frets(frets):
    fretted = [f for f in frets if f > 0]
    return float(np.mean(fretted)) if fretted else 0.0

def awkward_fingering_penalty(position, hand_center):
    f = position['fret']
    if f == 0: return 0.0
    return max(0.0, abs(f - hand_center) - 2) * 0.5

def context_cost(group_notes, group_positions):
    cost = 0.0
    for n, _ in zip(group_notes, group_positions):
        if n.get('in_chord') is False: cost += 0.15
        if n.get('in_key')   is False: cost += 0.10
    return cost

def enrich_candidate(candidate):
    positions = candidate['positions']
    frets   = [p['fret']   for p in positions]
    strings = [p['string'] for p in positions]
    candidate['center']       = estimate_hand_position_from_frets(frets)
    candidate['avg_string']   = float(np.mean(strings)) if strings else 0.0
    candidate['is_single']    = len(positions) == 1
    candidate['single_fret']  = positions[0]['fret']   if len(positions) == 1 else np.nan
    candidate['single_string']= positions[0]['string'] if len(positions) == 1 else np.nan
    return candidate

# Flat position prior — no GuitarSet records needed for a single debug file
POSITION_PRIOR_COSTS = {}
def position_prior_cost(midi, position):
    return float(POSITION_PRIOR_COSTS.get((int(midi), int(position['string']), int(position['fret'])), 0.0))

def group_notes_by_onset(notes, tolerance=ONSET_TOLERANCE_SECONDS):
    if not notes: return []
    notes_sorted = sorted(notes, key=lambda x: (x['start'], x['midi']))
    groups, current = [], [notes_sorted[0]]
    group_start = notes_sorted[0]['start']
    for n in notes_sorted[1:]:
        if abs(n['start'] - group_start) <= tolerance:
            current.append(n)
        else:
            groups.append(current)
            current = [n]
            group_start = n['start']
    groups.append(current)
    return groups

print('Shared cost helpers ready.')

Shared cost helpers ready.


In [7]:
# ── Cell 6: CAGED-box candidate generator ────────────────────────────────────
# (caged_voiced builds on top of this)

COMFORTABLE_CHORD_SPAN, MAX_CHORD_SPAN = 3, 4

def group_playability_cost_caged(gp):
    if not gp: return 0.0
    strings = [p['string'] for p in gp]; frets = [p['fret'] for p in gp]
    fretted = [f for f in frets if f > 0]
    if len(strings) != len(set(strings)): return float('inf')
    cost = 0.0; span = group_span(frets)
    if span > COMFORTABLE_CHORD_SPAN: cost += 2.0 * (span - COMFORTABLE_CHORD_SPAN)
    if span > MAX_CHORD_SPAN:         cost += 25.0 * (span - MAX_CHORD_SPAN)
    if fretted and min(fretted) <= 2 and max(fretted) >= 9: cost += 8.0
    if len(strings) >= 2:
        ss = max(strings) - min(strings)
        if ss > 4 and len(strings) <= 3: cost += 1.5 * (ss - 4)
    hc = estimate_hand_position_from_frets(frets)
    cost += sum(awkward_fingering_penalty(p, hc) for p in gp)
    if any(f == 0 for f in frets) and fretted and max(fretted) > 7: cost += 3.0
    return cost

BOX_WINDOW, SHIFT_FREE = 4, 2
BOX_CENTER_COST, BOX_OUTSIDE_COST, OPEN_OUT_OF_BOX_COST = 0.15, 3.00, 0.60
BOX_OFFBOX_COST, BOX_NONHOME_COST, BOX_LOWNECK_COST = 0.60, 1.00, 0.04
CAGED_WEIGHTS = {'playability': 0.80, 'context': 0.30, 'box_window': 1.00,
                 'hand_move': 0.70, 'position_prior': 1.00, 'voicing': 1.0}
PENTATONIC = {'major': [0,2,4,7,9], 'minor': [0,3,5,7,10]}
LOW_E_PC = OPEN_STRING_MIDI[0] % 12
SOLO_MOVE_SCALE = 0.35
SOLO_BOX_SCALE  = 0.30

def box_anchors_for_key(key, max_fret=MAX_FRET, window=BOX_WINDOW):
    rng = range(0, max_fret - window + 1)
    if key is None:
        return [{'anchor': a, 'key_cost': BOX_LOWNECK_COST * a} for a in rng]
    r = key['root_pc']; penta = PENTATONIC.get(key['mode'], PENTATONIC['minor'])
    box = set()
    for deg in penta:
        f = (deg + (r - LOW_E_PC)) % 12
        while f <= max_fret - 1: box.add(f); f += 12
    home = set(); h = (r - LOW_E_PC) % 12
    while h <= max_fret - 1: home.add(h); h += 12
    out = []
    for a in rng:
        d = min((abs(a - b) for b in box), default=0)
        kc = BOX_OFFBOX_COST * d + (0.0 if a in home else BOX_NONHOME_COST) + BOX_LOWNECK_COST * a
        out.append({'anchor': a, 'key_cost': kc})
    return out

def position_window_cost(p, anchor, window=BOX_WINDOW):
    f = p['fret']
    if f == 0: return 0.0 if anchor <= 2 else OPEN_OUT_OF_BOX_COST
    if anchor <= f <= anchor + window: return BOX_CENTER_COST * abs(f - (anchor + window / 2.0))
    return BOX_OUTSIDE_COST * ((anchor - f) if f < anchor else (f - (anchor + window)))

def candidate_window_cost(c, anchor):
    return sum(position_window_cost(p, anchor) for p in c['positions'])

def candidate_groups_caged(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    pls = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos: return []
        pls.append(pos)

    def _greedy(cost):
        used = set(); pick = {}
        for i in sorted(range(len(group_notes)), key=lambda k: -group_notes[k]['midi']):
            opts = sorted(pls[i], key=lambda p: (p['fret'], p['string']))
            chosen = next((p for p in opts if p['string'] not in used), opts[0])
            used.add(chosen['string']); pick[i] = chosen
        return enrich_candidate({'positions': [pick[i] for i in range(len(group_notes))], 'base_cost': cost})

    space = 1
    for pl in pls: space *= len(pl)
    if space > 20000:
        return [_greedy(10.0)]

    cands = []
    for combo in product(*pls):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo): continue
        play  = group_playability_cost_caged(combo)
        if not math.isfinite(play): continue
        prior = float(np.mean([position_prior_cost(n['midi'], p) for n, p in zip(group_notes, combo)]))
        base  = (CAGED_WEIGHTS['playability'] * play
               + CAGED_WEIGHTS['context']     * context_cost(group_notes, combo)
               + CAGED_WEIGHTS['position_prior'] * prior)
        cands.append(enrich_candidate({'positions': combo, 'base_cost': float(base)}))

    if not cands:
        cands.append(_greedy(100.0))

    return sorted(cands, key=lambda c: c['base_cost'])[:max_candidates]

print('CAGED-box candidate generator ready.')

CAGED-box candidate generator ready.


In [8]:
# ── Cell 7: CAGED-voiced voicing shapes + assign function ────────────────────

VOICING_SHAPES = [
    {'name':'E-maj',  'offsets':{0:0,1:2,2:2,3:1,4:0,5:0}, 'power':False},
    {'name':'A-maj',  'offsets':{1:0,2:2,3:2,4:2,5:0},     'power':False},
    {'name':'D-maj',  'offsets':{2:0,3:2,4:3,5:2},         'power':False},
    {'name':'C-maj',  'offsets':{1:3,2:2,3:0,4:1,5:0},     'power':False},
    {'name':'G-maj',  'offsets':{0:3,1:2,2:0,3:0,4:0,5:3}, 'power':False},
    {'name':'E-min',  'offsets':{0:0,1:2,2:2,3:0,4:0,5:0}, 'power':False},
    {'name':'A-min',  'offsets':{1:0,2:2,3:2,4:1,5:0},     'power':False},
    {'name':'E-7',    'offsets':{0:0,1:2,2:0,3:1,4:0,5:0}, 'power':False},
    {'name':'A-7',    'offsets':{1:0,2:2,3:0,4:2,5:0},     'power':False},
    {'name':'E-m7',   'offsets':{0:0,1:2,2:0,3:0,4:0,5:0}, 'power':False},
    {'name':'A-m7',   'offsets':{1:0,2:2,3:0,4:1,5:0},     'power':False},
    {'name':'Emaj7',  'offsets':{0:0,1:2,2:1,3:1,4:0,5:0}, 'power':False},
    {'name':'Amaj7',  'offsets':{1:0,2:2,3:1,4:2,5:0},     'power':False},
    {'name':'5-E',    'offsets':{0:0,1:2},                  'power':True},
    {'name':'5-A',    'offsets':{1:0,2:2},                  'power':True},
    {'name':'5-D',    'offsets':{2:0,3:2},                  'power':True},
    {'name':'5-E-oct','offsets':{0:0,1:2,2:2},              'power':True},
    {'name':'5-A-oct','offsets':{1:0,2:2,3:2},              'power':True},
    {'name':'5-D-oct','offsets':{2:0,3:2,4:2},              'power':True},
    {'name':'oct-E',  'offsets':{0:0,2:2},                  'power':True},
    {'name':'oct-A',  'offsets':{1:0,3:2},                  'power':True},
]

def _off_from_min(d):
    m = min(d.values()); return {k: v - m for k, v in d.items()}

def voicing_bonus(positions):
    pts = {p['string']: p['fret'] for p in positions}
    strings = sorted(pts)
    if len(strings) < 2: return 0.0
    cand_off = _off_from_min(pts); n = len(strings); best = 0.0
    for sh in VOICING_SHAPES:
        if (n < 2) if sh['power'] else (n < 3): continue
        smap = sh['offsets']
        if not all(s in smap for s in strings): continue
        if _off_from_min({s: smap[s] for s in strings}) == cand_off:
            best = max(best, 0.6 + 0.4 * (n / len(smap)))
    return best

def candidate_groups_voiced(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    pool = candidate_groups_caged(group_notes, max_candidates=max(max_candidates * 3, 24))
    w = CAGED_WEIGHTS.get('voicing', 0.0)
    if w and len(group_notes) >= 2:
        for c in pool:
            b = voicing_bonus(c['positions'])
            if b: c['base_cost'] = c['base_cost'] - w * b
        pool = sorted(pool, key=lambda c: c['base_cost'])
    return pool[:max_candidates]

def assign_caged_voiced(notes, key=None):
    groups = group_notes_by_onset(notes)
    if not groups: return []
    allc      = [candidate_groups_voiced(g) for g in groups]
    is_chord  = [len(g) >= 2 for g in groups]
    anchors   = box_anchors_for_key(key); A = len(anchors)
    af        = np.array([a['anchor'] for a in anchors], dtype=float)
    n         = len(groups)
    ec        = np.empty((n, A)); ecand = [[0] * A for _ in range(n)]

    for i, cands in enumerate(allc):
        if not cands:
            # fallback: treat note as unassignable, skip
            ec[i, :] = 999.0; continue
        kc_scale = 1.0 if is_chord[i] else SOLO_BOX_SCALE
        for j, anc in enumerate(anchors):
            a = anc['anchor']; best = None; bci = 0
            for ci, c in enumerate(cands):
                tot = c['base_cost'] + CAGED_WEIGHTS['box_window'] * candidate_window_cost(c, a)
                if best is None or tot < best: best, bci = tot, ci
            ec[i, j] = best + kc_scale * anc['key_cost']; ecand[i][j] = bci

    delta      = np.abs(af[:, None] - af[None, :])
    base_trans = (CAGED_WEIGHTS['hand_move'] * np.maximum(delta - SHIFT_FREE, 0.0)
                  + 0.5 * np.maximum(delta - LARGE_JUMP_THRESHOLD, 0.0) ** 2)
    dp   = np.empty((n, A)); back = np.zeros((n, A), dtype=int)
    dp[0] = ec[0]; back[0] = -1
    for i in range(1, n):
        chordy     = is_chord[i] or is_chord[i - 1]
        step_trans = base_trans if chordy else base_trans * SOLO_MOVE_SCALE
        scores     = dp[i - 1][:, None] + step_trans + ec[i][None, :]
        back[i]    = np.argmin(scores, axis=0)
        dp[i]      = scores[back[i], np.arange(A)]

    j = int(np.argmin(dp[-1])); chosen = [j]
    for i in range(n - 1, 0, -1): j = int(back[i][j]); chosen.append(j)
    chosen = list(reversed(chosen))

    pred = []
    for i, (g, cands) in enumerate(zip(groups, allc)):
        if not cands: continue
        c = cands[ecand[i][chosen[i]]]
        for note, p in zip(g, c['positions']):
            row = dict(note)
            row.update({'string': p['string'], 'fret': p['fret'],
                        'string_name': p['string_name'],
                        'anchor': anchors[chosen[i]]['anchor']})
            pred.append(row)
    return sorted(pred, key=lambda x: (x['start'], x['midi']))

print('CAGED-voiced ready. VOICING_SHAPES:', len(VOICING_SHAPES))

CAGED-voiced ready. VOICING_SHAPES: 21


In [9]:
# ── Cell 8: Run Basic Pitch ───────────────────────────────────────────────────
from basic_pitch.inference import predict as basic_pitch_predict

audio_path = Path(AUDIO_FILE)
assert audio_path.exists(), f'File not found: {audio_path}'

print(f'Running Basic Pitch on: {audio_path.name} ...')
_, _, note_events = basic_pitch_predict(
    str(audio_path),
    onset_threshold=ONSET_THRESHOLD,
    minimum_frequency=OPEN_STRING_MIDI[0],
)

raw_notes = []
for (start_t, end_t, pitch_midi, amplitude, _) in note_events:
    if amplitude < BASIC_PITCH_AMPLITUDE_THRESHOLD: continue
    midi = int(round(pitch_midi))
    if not (BASIC_PITCH_MIN_MIDI <= midi <= BASIC_PITCH_MAX_MIDI): continue
    raw_notes.append({
        'start':      float(start_t),
        'duration':   float(end_t - start_t),
        'midi':       midi,
        'pitch_class': midi % 12,
        'amplitude':  float(amplitude),
        # context fields (flat — no chord detection here)
        'in_key':   None,
        'in_chord': None,
        'key_label': KEY_LABEL,
    })

raw_notes.sort(key=lambda n: n['start'])
print(f'\nBasic Pitch detected {len(raw_notes)} notes after filtering.')
print('\n── RAW MIDI NOTES (what Basic Pitch sees) ──────────────────────')
rows = []
for n in raw_notes:
    rows.append({
        'time (s)':  round(n['start'], 3),
        'dur (s)':   round(n['duration'], 3),
        'MIDI':      n['midi'],
        'note':      midi_to_note_name(n['midi']),
        'amp':       round(n['amplitude'], 3),
        'all positions': ', '.join(
            f"{STRING_NAMES[p['string']]}@{p['fret']}"
            for p in get_possible_positions(n['midi'])
        )
    })
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.max_rows', 300)
display(pd.DataFrame(rows))

Running Basic Pitch on: isThisItIntro.mp3 ...
Predicting MIDI for /content/drive/MyDrive/Capstone/Audio/isThisItIntro.mp3...

Basic Pitch detected 47 notes after filtering.

── RAW MIDI NOTES (what Basic Pitch sees) ──────────────────────


,time (s),dur (s),MIDI,note,amp,all positions
0,0.639,0.627,65,F4,0.745,"A@20, D@15, G@10, B@6, high_E@1"
1,0.964,0.615,69,A4,0.571,"A@24, D@19, G@14, B@10, high_E@5"
2,1.265,0.604,65,F4,0.632,"A@20, D@15, G@10, B@6, high_E@1"
3,1.579,0.617,69,A4,0.595,"A@24, D@19, G@14, B@10, high_E@5"
4,1.892,0.930,64,E4,0.536,"low_E@24, A@19, D@14, G@9, B@5, high_E@0"
5,1.916,0.280,45,A2,0.483,"low_E@5, A@0"
6,2.196,0.302,45,A2,0.549,"low_E@5, A@0"
7,2.509,0.604,62,D4,0.553,"low_E@22, A@17, D@12, G@7, B@3"
8,2.811,0.627,69,A4,0.585,"A@24, D@19, G@14, B@10, high_E@5"
9,2.823,0.290,50,D3,0.487,"low_E@10, A@5, D@0"


In [10]:
# ── Cell 9: Run caged_voiced assignment ──────────────────────────────────────
key = parse_key(KEY_LABEL)
notes_in_range = [n for n in raw_notes if get_possible_positions(n['midi'])]
assigned = assign_caged_voiced(notes_in_range, key=key)

print(f'Assigned {len(assigned)} notes (key: {KEY_LABEL or "unknown/flat"}).\n')
print('── ASSIGNED NOTES ──────────────────────────────────────────────')
assign_rows = []
for n in assigned:
    assign_rows.append({
        'time (s)':  round(n['start'], 3),
        'MIDI':      n['midi'],
        'note':      midi_to_note_name(n['midi']),
        'string':    STRING_NAMES[n['string']],
        'fret':      n['fret'],
        'box anchor': n.get('anchor', '?'),
    })
display(pd.DataFrame(assign_rows))

Assigned 47 notes (key: unknown/flat).

── ASSIGNED NOTES ──────────────────────────────────────────────


,time (s),MIDI,note,string,fret,box anchor
0,0.639,65,F4,B,6,4
1,0.964,69,A4,high_E,5,3
2,1.265,65,F4,B,6,4
3,1.579,69,A4,high_E,5,3
4,1.892,64,E4,high_E,0,1
5,1.916,45,A2,A,0,1
6,2.196,45,A2,A,0,0
7,2.509,62,D4,B,3,1
8,2.811,69,A4,high_E,5,3
9,2.823,50,D3,A,5,3


In [11]:
# ── Cell 10: ASCII tab ────────────────────────────────────────────────────────
DISPLAY_TO_STRING = [5, 4, 3, 2, 1, 0]  # high-e on top
STRING_LABELS     = ['e', 'B', 'G', 'D', 'A', 'E']

def render_ascii_tab(notes, tempo_bpm=ASSUMED_TEMPO_BPM,
                     subdivisions_per_beat=SUBDIVISIONS_PER_BEAT,
                     beats_per_measure=BEATS_PER_MEASURE,
                     measures_per_line=MEASURES_PER_LINE,
                     col_width=COL_WIDTH):
    if not notes:
        print('(no notes)'); return
    notes = sorted(notes, key=lambda n: (n['start'], n['midi']))
    end_time = max(n['start'] + n.get('duration', 0.5) for n in notes) + 0.5
    step  = 60.0 / tempo_bpm / subdivisions_per_beat
    grid  = [i * step for i in range(int(end_time / step) + subdivisions_per_beat + 2)]
    n_cols = len(grid)
    cells = [[None] * n_cols for _ in range(6)]
    collisions = 0
    for note in notes:
        s = note.get('string'); f = note.get('fret')
        if s is None or f is None or s not in DISPLAY_TO_STRING: continue
        col = min(range(n_cols), key=lambda i: abs(grid[i] - note['start']))
        row = DISPLAY_TO_STRING.index(s)
        if cells[row][col] is not None: collisions += 1
        cells[row][col] = f

    def fmt(v):
        if v is None: return '-' * col_width
        s = str(v)
        return s + '-' * (col_width - len(s)) if len(s) < col_width else s[:col_width]

    formatted      = [[fmt(cells[r][c]) for c in range(n_cols)] for r in range(6)]
    cols_per_measure = beats_per_measure * subdivisions_per_beat
    cols_per_line    = cols_per_measure * measures_per_line
    chunk = 0
    while chunk < n_cols:
        end = min(chunk + cols_per_line, n_cols)
        for ri in range(6):
            row_str = ''
            for ci in range(chunk, end):
                row_str += formatted[ri][ci]
                if (ci - chunk + 1) % cols_per_measure == 0 and ci != end - 1:
                    row_str += '|'
            print(f'{STRING_LABELS[ri]}|{row_str}|')
        print()
        chunk = end
    if collisions:
        print(f'[{collisions} collision(s) in grid; later note shown]')

print('=' * 60)
print(f'CAGED-VOICED TAB — {Path(AUDIO_FILE).name}')
print(f'Key: {KEY_LABEL or "unknown"}')
print('=' * 60)
render_ascii_tab(assigned)

CAGED-VOICED TAB — isThisItIntro.mp3
Key: unknown
e|----------------5-------5-------|0-----------5-----------5-------|5-------0---0-------5-------6---|6-------6-------0---6-------6---|
B|------------6-------6-----------|--------3-------6-----------6---|----------------3-----------6---|----6---------------------------|
G|--------------------------------|--------------------------------|--------------------------------|------------------------7-------|
D|--------------------------------|--------------------------------|--------------------------------|--------8-----------------------|
A|--------------------------------|0---0-------5-------------------|--------0---0-------5-----------|--------------------------------|
E|--------------------------------|--------------------------------|--------------------------------|6---6---6-----------------------|

e|--------6-----------6---6---6---|----6---------------------------|
B|----6-------6-----------5-------|--------------------------------|
G

In [12]:
# ── Cell 11 (optional): Pitch histogram ──────────────────────────────────────
# Quick sanity check — do detected pitches match what you actually played?
from collections import Counter
pitch_counts = Counter(midi_to_note_name(n['midi']) for n in raw_notes)
print('Detected pitch distribution (most common first):')
for note, count in pitch_counts.most_common():
    print(f'  {note:>4}  {"█" * count}  ({count})')

Detected pitch distribution (most common first):
   A#4  ██████████  (10)
    F4  ████████  (8)
   A#2  ███████  (7)
    A4  ██████  (6)
    E4  █████  (5)
    A2  ████  (4)
    D4  ████  (4)
    D3  ██  (2)
   A#3  █  (1)


In [13]:
# ── Cell 12: Sweep KEY_LABEL × BOX_LOWNECK_COST and compare tabs ──────────────
# Reruns caged_voiced under a grid of settings using the SAME raw_notes,
# so you can see which combination recovers the natural open-position fingering.
# Does not mutate global state — restores originals when done.

import io
from contextlib import redirect_stdout

KEY_SWEEP      = [None, 'C major', 'G major']
LOWNECK_SWEEP  = [0.04, 0.10, 0.15]   # BOX_LOWNECK_COST values to try

_orig_lowneck = BOX_LOWNECK_COST

def _mean_fret(assigned):
    fretted = [n['fret'] for n in assigned if n.get('fret', 0) > 0]
    return float(np.mean(fretted)) if fretted else 0.0

def _render_to_str(notes):
    buf = io.StringIO()
    with redirect_stdout(buf):
        render_ascii_tab(notes)
    return buf.getvalue()

results = []
for lowneck in LOWNECK_SWEEP:
    # patch the module-level constant used inside box_anchors_for_key
    globals()['BOX_LOWNECK_COST'] = lowneck
    for key_label in KEY_SWEEP:
        key = parse_key(key_label)
        notes_in = [dict(n) for n in raw_notes if get_possible_positions(n['midi'])]
        for n in notes_in:
            n['key_label'] = key_label
        assigned_s = assign_caged_voiced(notes_in, key=key)
        results.append({
            'key': key_label or 'None',
            'lowneck': lowneck,
            'mean_fret': round(_mean_fret(assigned_s), 2),
            'max_fret': max((n['fret'] for n in assigned_s), default=0),
            'assigned': assigned_s,
        })

# restore original
globals()['BOX_LOWNECK_COST'] = _orig_lowneck

# ── Summary table: lower mean_fret = closer to open position ─────────────────
print('Summary — lower mean_fret means more open-position (Ode to Joy wants ~2-3):\n')
summary = pd.DataFrame([{k: r[k] for k in ['key', 'lowneck', 'mean_fret', 'max_fret']} for r in results])
display(summary.sort_values(['mean_fret', 'max_fret']).reset_index(drop=True))

# ── Print each tab ───────────────────────────────────────────────────────────
for r in results:
    print('\n' + '=' * 64)
    print(f"KEY = {r['key']:<10}  BOX_LOWNECK_COST = {r['lowneck']:<5}  "
          f"mean_fret = {r['mean_fret']}  max_fret = {r['max_fret']}")
    print('=' * 64)
    print(_render_to_str(r['assigned']))


Summary — lower mean_fret means more open-position (Ode to Joy wants ~2-3):



,key,lowneck,mean_fret,max_fret
0,C major,0.15,4.62,6
1,None,0.15,4.84,8
2,G major,0.04,5.66,7
3,G major,0.10,5.66,7
4,G major,0.15,5.66,7
5,None,0.04,5.72,8
6,None,0.10,5.72,8
7,C major,0.10,8.27,12
8,C major,0.04,8.39,12



KEY = None        BOX_LOWNECK_COST = 0.04   mean_fret = 5.72  max_fret = 8
e|----------------5-------5-------|0-----------5-----------5-------|5-------0---0-------5-------6---|6-------6-------0---6-------6---|
B|------------6-------6-----------|--------3-------6-----------6---|----------------3-----------6---|----6---------------------------|
G|--------------------------------|--------------------------------|--------------------------------|------------------------7-------|
D|--------------------------------|--------------------------------|--------------------------------|--------8-----------------------|
A|--------------------------------|0---0-------5-------------------|--------0---0-------5-----------|--------------------------------|
E|--------------------------------|--------------------------------|--------------------------------|6---6---6-----------------------|

e|--------6-----------6---6---6---|----6---------------------------|
B|----6-------6-----------5-------|---------

In [14]:
# ── Cell 13: Scale-aware / chord-gated home box variant ──────────────────────
# Problem: box_anchors_for_key() defines "home" as where the key root sits on the
# low-E string. For single-note melodies that drags everything up the neck for
# keys whose root is high on low-E (C->fret8, D->fret10, etc.). A real player
# just plays a melody in open/first position regardless of key.
#
# Fix: gate the home-box pull on whether the onset is a chord.
#   - CHORDS keep the original root-position home box (barre logic is correct there)
#   - SINGLE NOTES get NO non-home penalty and a real low-neck preference,
#     so they default to the lowest comfortable window.
#
# This is a drop-in re-implementation of assign_caged_voiced that takes a
# per-anchor key_cost that depends on is_chord. Originals are left untouched.

SINGLE_NOTE_LOWNECK_COST = 0.15   # low-neck ramp applied to single notes
SINGLE_NOTE_NONHOME      = 0.0    # non-home penalty for single notes (0 = ignore key center)

def box_anchors_for_key_gated(key, is_chord_onset, max_fret=MAX_FRET, window=BOX_WINDOW):
    """Like box_anchors_for_key, but for single-note onsets it drops the
    non-home penalty and applies a stronger low-neck preference."""
    rng = range(0, max_fret - window + 1)

    if not is_chord_onset:
        # SINGLE NOTE: ignore key-center home box; just prefer the low neck.
        if key is None:
            return [{'anchor': a, 'key_cost': SINGLE_NOTE_LOWNECK_COST * a} for a in rng]
        r = key['root_pc']; penta = PENTATONIC.get(key['mode'], PENTATONIC['minor'])
        box = set()
        for deg in penta:
            f = (deg + (r - LOW_E_PC)) % 12
            while f <= max_fret - 1: box.add(f); f += 12
        out = []
        for a in rng:
            d = min((abs(a - b) for b in box), default=0)
            # keep a gentle in-scale-box nudge, but NO non-home term, strong lowneck
            kc = BOX_OFFBOX_COST * d + SINGLE_NOTE_NONHOME + SINGLE_NOTE_LOWNECK_COST * a
            out.append({'anchor': a, 'key_cost': kc})
        return out

    # CHORD: original root-position home box (unchanged).
    return box_anchors_for_key(key, max_fret=max_fret, window=window)


def assign_caged_voiced_gated(notes, key=None):
    """assign_caged_voiced with chord-gated home-box anchoring.

    Each onset gets its own anchor cost vector depending on whether it is a
    chord (>=2 notes) or a single note.
    """
    groups = group_notes_by_onset(notes)
    if not groups: return []
    allc     = [candidate_groups_voiced(g) for g in groups]
    is_chord = [len(g) >= 2 for g in groups]
    n        = len(groups)

    # Build a per-onset anchor set. Anchor list length A is identical across
    # onsets (same fret range), so transitions still line up.
    anchor_sets = [box_anchors_for_key_gated(key, is_chord[i]) for i in range(n)]
    A  = len(anchor_sets[0])
    af = np.array([a['anchor'] for a in anchor_sets[0]], dtype=float)

    ec    = np.empty((n, A)); ecand = [[0] * A for _ in range(n)]
    for i, cands in enumerate(allc):
        if not cands:
            ec[i, :] = 999.0; continue
        kc_scale = 1.0 if is_chord[i] else SOLO_BOX_SCALE
        anchors  = anchor_sets[i]
        for j, anc in enumerate(anchors):
            a = anc['anchor']; best = None; bci = 0
            for ci, c in enumerate(cands):
                tot = c['base_cost'] + CAGED_WEIGHTS['box_window'] * candidate_window_cost(c, a)
                if best is None or tot < best: best, bci = tot, ci
            ec[i, j] = best + kc_scale * anc['key_cost']; ecand[i][j] = bci

    delta      = np.abs(af[:, None] - af[None, :])
    base_trans = (CAGED_WEIGHTS['hand_move'] * np.maximum(delta - SHIFT_FREE, 0.0)
                  + 0.5 * np.maximum(delta - LARGE_JUMP_THRESHOLD, 0.0) ** 2)
    dp   = np.empty((n, A)); back = np.zeros((n, A), dtype=int)
    dp[0] = ec[0]; back[0] = -1
    for i in range(1, n):
        chordy     = is_chord[i] or is_chord[i - 1]
        step_trans = base_trans if chordy else base_trans * SOLO_MOVE_SCALE
        scores     = dp[i - 1][:, None] + step_trans + ec[i][None, :]
        back[i]    = np.argmin(scores, axis=0)
        dp[i]      = scores[back[i], np.arange(A)]

    j = int(np.argmin(dp[-1])); chosen = [j]
    for i in range(n - 1, 0, -1): j = int(back[i][j]); chosen.append(j)
    chosen = list(reversed(chosen))

    pred = []
    for i, (g, cands) in enumerate(zip(groups, allc)):
        if not cands: continue
        c = cands[ecand[i][chosen[i]]]
        for note, p in zip(g, c['positions']):
            row = dict(note)
            row.update({'string': p['string'], 'fret': p['fret'],
                        'string_name': p['string_name'],
                        'anchor': anchor_sets[i][chosen[i]]['anchor']})
            pred.append(row)
    return sorted(pred, key=lambda x: (x['start'], x['midi']))


# ── A/B: original (best lowneck hack) vs gated variant, across keys ──────────
import io
from contextlib import redirect_stdout

def _mean_fret(assigned):
    fretted = [n['fret'] for n in assigned if n.get('fret', 0) > 0]
    return float(np.mean(fretted)) if fretted else 0.0

def _render_to_str(notes):
    buf = io.StringIO()
    with redirect_stdout(buf):
        render_ascii_tab(notes)
    return buf.getvalue()

_orig_lowneck = BOX_LOWNECK_COST
KEYS = [None, 'C major', 'G major']

print('A/B — original (lowneck=0.15) vs chord-gated home box:\n')
ab_rows = []
ab_tabs = []
for key_label in KEYS:
    key = parse_key(key_label)
    notes_in = [dict(n) for n in raw_notes if get_possible_positions(n['midi'])]
    for nn in notes_in: nn['key_label'] = key_label

    # ORIGINAL with the lowneck hack at 0.15
    globals()['BOX_LOWNECK_COST'] = 0.15
    a_orig = assign_caged_voiced(notes_in, key=key)
    globals()['BOX_LOWNECK_COST'] = _orig_lowneck

    # GATED variant (lowneck hack OFF -> default 0.04)
    a_gated = assign_caged_voiced_gated(notes_in, key=key)

    ab_rows.append({'key': key_label or 'None',
                    'orig_lowneck0.15_mean': round(_mean_fret(a_orig), 2),
                    'orig_max': max((n['fret'] for n in a_orig), default=0),
                    'gated_mean': round(_mean_fret(a_gated), 2),
                    'gated_max': max((n['fret'] for n in a_gated), default=0)})
    ab_tabs.append((key_label or 'None', a_orig, a_gated))

display(pd.DataFrame(ab_rows))

for key_label, a_orig, a_gated in ab_tabs:
    print('\n' + '=' * 64)
    print(f'KEY = {key_label}')
    print('=' * 64)
    print('--- ORIGINAL (lowneck=0.15) ---')
    print(_render_to_str(a_orig))
    print('--- GATED variant ---')
    print(_render_to_str(a_gated))


A/B — original (lowneck=0.15) vs chord-gated home box:



,key,orig_lowneck0.15_mean,orig_max,gated_mean,gated_max
0,None,4.84,8,5.72,8
1,C major,4.62,6,5.47,8
2,G major,5.66,7,5.33,6



KEY = None
--- ORIGINAL (lowneck=0.15) ---
e|----------------5-------5-------|0-----------5-----------5-------|5-------0---0-------5-------6---|6-------6-------0---6-------6---|
B|------------6-------6-----------|--------3-------6-----------6---|----------------3-----------6---|----6-------------------3-------|
G|--------------------------------|--------------------------------|--------------------------------|--------------------------------|
D|--------------------------------|------------0-------------------|--------------------0-----------|--------8-----------------------|
A|--------------------------------|0---0---------------------------|--------0---0-------------------|--------------------------------|
E|--------------------------------|--------------------------------|--------------------------------|6---6---6-----------------------|

e|----1---6---1-------6---6---6---|----6---------------------------|
B|------------------------5-------|--------------------------------|
G|-----

In [18]:
# ── Cell 15: proximity as a Viterbi transition term (joint, not greedy) ──────
# Replaces the broken greedy re-pick. Proximity now lives INSIDE the decode as a
# fret-distance penalty between consecutive single-note placements, optimized
# jointly over the phrase. The box-anchor + lowneck terms are KEPT, so there's
# still gravity toward low/sane positions — that's what prevents the runaway to
# frets 19-22 you just saw.
#
# This is a full re-decode at the CANDIDATE level (not the anchor level), so it
# can express "use B-string fret 5 for E4 because the neighbors are at 5-6".

SINGLE_PROX_WEIGHT = 0.85   # cost per fret of jump between consecutive single notes
OPEN_DEFECT_COST   = 1.20   # extra cost to use an open string when prev was fretted high
LOWNECK_PULL       = 0.10   # gentle pull toward low frets (keeps phrases from floating up)

def assign_voiced_prox_viterbi(notes, key=None):
    groups   = group_notes_by_onset(notes)
    if not groups:
        return []
    allc     = [candidate_groups_voiced(g) for g in groups]
    is_chord = [len(g) >= 2 for g in groups]
    n = len(groups)

    # Emission cost per candidate = its base cost + a gentle lowneck pull on the
    # candidate's mean fret. (Box-window handled inside base via candidate_groups.)
    emis = []
    for gi, cands in enumerate(allc):
        if not cands:
            emis.append([(None, 0.0)])
            continue
        row = []
        for c in cands:
            frets = [p['fret'] for p in c['positions']]
            mean_fret = np.mean([f for f in frets if f > 0]) if any(f>0 for f in frets) else 0.0
            e = c['base_cost'] + LOWNECK_PULL * mean_fret
            row.append((c, e))
        emis.append(row)

    def trans(prev_c, curr_c, prev_chord, curr_chord):
        if prev_c is None or curr_c is None:
            return 0.0
        # hand-center distance (works for chords and single notes)
        pf = [p['fret'] for p in prev_c['positions'] if p['fret'] > 0]
        cf = [p['fret'] for p in curr_c['positions'] if p['fret'] > 0]
        pc = np.mean(pf) if pf else 0.0
        cc = np.mean(cf) if cf else 0.0
        cost = CAGED_WEIGHTS['hand_move'] * abs(cc - pc)
        # single-note proximity: the key term for the E4 open-vs-Bstring choice
        if not prev_chord and not curr_chord:
            pfret = prev_c['positions'][0]['fret']
            cfret = curr_c['positions'][0]['fret']
            if cfret == 0 and pc > 3:
                cost += OPEN_DEFECT_COST       # don't defect to open mid-phrase
            else:
                cost += SINGLE_PROX_WEIGHT * abs(cfret - pfret)
        return cost

    # Viterbi over candidates
    dp   = [[math.inf]*len(emis[i]) for i in range(n)]
    back = [[-1]*len(emis[i]) for i in range(n)]
    for ci, (_, e) in enumerate(emis[0]):
        dp[0][ci] = e
    for i in range(1, n):
        for ci, (cc, ce) in enumerate(emis[i]):
            best, bp = math.inf, -1
            for pi, (pc, _) in enumerate(emis[i-1]):
                t = trans(pc, cc, is_chord[i-1], is_chord[i])
                tot = dp[i-1][pi] + t + ce
                if tot < best:
                    best, bp = tot, pi
            dp[i][ci] = best; back[i][ci] = bp

    # traceback
    j = int(np.argmin(dp[-1])); path = [j]
    for i in range(n-1, 0, -1):
        j = back[i][j]; path.append(j)
    path.reverse()

    out = []
    for gi, (g, ci) in enumerate(zip(groups, path)):
        c = emis[gi][ci][0]
        if c is None:
            continue
        for note, p in zip(g, c['positions']):
            row = dict(note); row.update({'string': p['string'], 'fret': p['fret'],
                        'string_name': p['string_name']})
            out.append(row)
    return sorted(out, key=lambda x: (x['start'], x['midi']))


# ── A/B vs gated ─────────────────────────────────────────────────────────────
import io
from contextlib import redirect_stdout
def _render(a):
    b = io.StringIO()
    with redirect_stdout(b): render_ascii_tab(a)
    return b.getvalue()
def _mf(a):
    f=[n['fret'] for n in a if n.get('fret',0)>0]; return round(float(np.mean(f)),2) if f else 0.0
def _open(a): return sum(1 for n in a if n['fret']==0)

key = parse_key(KEY_LABEL)
in_range = [n for n in raw_notes if get_possible_positions(n['midi'])]
a_gated = assign_caged_voiced_gated(in_range, key=key)
a_prox  = assign_voiced_prox_viterbi(in_range, key=key)

print(f"gated         : mean_fret={_mf(a_gated):5}  open={_open(a_gated)}  max={max((n['fret'] for n in a_gated),default=0)}")
print(f"prox-viterbi  : mean_fret={_mf(a_prox):5}  open={_open(a_prox)}  max={max((n['fret'] for n in a_prox),default=0)}")
print('\n' + '='*64 + '\nGATED\n' + '='*64)
print(_render(a_gated))
print('\n' + '='*64 + '\nPROX-VITERBI (joint)\n' + '='*64)
print(_render(a_prox))

gated         : mean_fret= 5.72  open=8  max=8
prox-viterbi  : mean_fret= 5.49  open=0  max=7

GATED
e|----------------5-------5-------|0-----------5-----------5-------|5-------0---0-------5-------6---|6-------6-------0---6-------6---|
B|------------6-------6-----------|--------3-------6-----------6---|----------------3-----------6---|----6---------------------------|
G|--------------------------------|--------------------------------|--------------------------------|------------------------7-------|
D|--------------------------------|--------------------------------|--------------------------------|--------8-----------------------|
A|--------------------------------|0---0-------5-------------------|--------0---0-------5-----------|--------------------------------|
E|--------------------------------|--------------------------------|--------------------------------|6---6---6-----------------------|

e|--------6-----------6---6---6---|----6---------------------------|
B|----6-------6----

In [19]:
# ── Cell 16: hand-span window transition cost ────────────────────────────────
# Replaces raw fret-distance proximity with a "movable 4-fret hand window".
# A note INSIDE the current window [lo, lo+SPAN] is free (any finger reaches it
# without moving the hand). A note OUTSIDE pays for the shift needed to bring the
# window to it. This is what distinguishes G-string fret 7 (in-window, pinky
# reach) from B-string fret 3 (below-window, whole-hand shift) when the hand is
# already at frets 5-6 — the two are equidistant in fret NUMBER but not to a hand.
#
# Window is tracked as an EMA over recent fretted notes (open strings don't move
# the hand). Same Viterbi as Cell 15; only the single-note transition term changes.

HAND_SPAN          = 3      # frets the four fingers cover beyond the index (index..index+3)
SHIFT_WEIGHT       = 0.85   # cost per fret of hand-window SHIFT (out-of-window notes)
INWINDOW_COST      = 0.05   # tiny cost for in-window notes (prefers center-of-hand slightly)
OPEN_DEFECT_COST   = 1.20   # using an open string while hand is up the neck
LOWNECK_PULL       = 0.12   # gentle gravity toward low positions (anti-runaway)
STRETCH_TOLERANCE  = 1      # index can reach back / pinky forward by this many frets cheaply

def _window_shift_cost(cand_fret, lo):
    """Cost to play cand_fret given hand window [lo, lo+HAND_SPAN]."""
    hi = lo + HAND_SPAN
    if lo <= cand_fret <= hi:
        # in-window: free-ish, tiny preference for not living at the extreme edges
        return INWINDOW_COST * 0  # flat zero inside; keep hook for tuning
    # outside: allow a cheap 1-fret stretch (index back / pinky forward), then pay to shift
    if cand_fret < lo:
        over = (lo - cand_fret)
    else:
        over = (cand_fret - hi)
    stretch = min(over, STRETCH_TOLERANCE)
    shift   = over - stretch
    return INWINDOW_COST * stretch + SHIFT_WEIGHT * shift

def assign_voiced_span_viterbi(notes, key=None):
    groups   = group_notes_by_onset(notes)
    if not groups:
        return []
    allc     = [candidate_groups_voiced(g) for g in groups]
    is_chord = [len(g) >= 2 for g in groups]
    n = len(groups)

    # Emission cost = base + gentle lowneck pull (gravity that prevents runaway).
    emis = []
    for cands in allc:
        if not cands:
            emis.append([(None, 0.0)]); continue
        row = []
        for c in cands:
            frets = [p['fret'] for p in c['positions'] if p['fret'] > 0]
            mean_fret = np.mean(frets) if frets else 0.0
            row.append((c, c['base_cost'] + LOWNECK_PULL * mean_fret))
        emis.append(row)

    # Hand-window low edge implied by a candidate (for single notes = the note's
    # fret minus a bit, so the note sits comfortably under a finger, not always
    # the index). We approximate window-lo as fret - 1 (note tends to be played
    # by index/middle), clamped at 0.
    def window_lo_of(cand):
        fr = [p['fret'] for p in cand['positions'] if p['fret'] > 0]
        if not fr:
            return None  # all-open: doesn't define a hand position
        return max(0, int(round(np.mean(fr))) - 1)

    def trans(prev_c, curr_c, prev_chord, curr_chord, prev_lo):
        if prev_c is None or curr_c is None:
            return 0.0, prev_lo
        # update running window-lo from previous placement
        plo = window_lo_of(prev_c)
        lo  = plo if plo is not None else prev_lo
        # hand-center move term (keeps chords sensible)
        pf = [p['fret'] for p in prev_c['positions'] if p['fret'] > 0]
        cf = [p['fret'] for p in curr_c['positions'] if p['fret'] > 0]
        pc = np.mean(pf) if pf else 0.0
        cc = np.mean(cf) if cf else 0.0
        cost = CAGED_WEIGHTS['hand_move'] * abs(cc - pc)
        if not prev_chord and not curr_chord:
            cfret = curr_c['positions'][0]['fret']
            if cfret == 0:
                # open string: free only if hand already near nut
                cost += 0.0 if (lo is None or lo <= 2) else OPEN_DEFECT_COST
            elif lo is not None:
                cost += _window_shift_cost(cfret, lo)
        new_lo = window_lo_of(curr_c)
        return cost, (new_lo if new_lo is not None else lo)

    # Viterbi; carry the running window-lo through the back-pointers via a per-state
    # estimate. We approximate by storing lo with each dp state.
    INF = math.inf
    dp   = [[INF]*len(emis[i]) for i in range(n)]
    lo_s = [[None]*len(emis[i]) for i in range(n)]
    back = [[-1]*len(emis[i]) for i in range(n)]
    for ci, (c, e) in enumerate(emis[0]):
        dp[0][ci] = e
        lo_s[0][ci] = window_lo_of(c) if c else None
    for i in range(1, n):
        for ci, (cc, ce) in enumerate(emis[i]):
            best, bp, blo = INF, -1, None
            for pi, (pc, _) in enumerate(emis[i-1]):
                t, nlo = trans(pc, cc, is_chord[i-1], is_chord[i], lo_s[i-1][pi])
                tot = dp[i-1][pi] + t + ce
                if tot < best:
                    best, bp, blo = tot, pi, nlo
            dp[i][ci] = best; back[i][ci] = bp; lo_s[i][ci] = blo

    j = int(np.argmin(dp[-1])); path = [j]
    for i in range(n-1, 0, -1):
        j = back[i][j]; path.append(j)
    path.reverse()

    out = []
    for gi, (g, ci) in enumerate(zip(groups, path)):
        c = emis[gi][ci][0]
        if c is None:
            continue
        for note, p in zip(g, c['positions']):
            row = dict(note); row.update({'string': p['string'], 'fret': p['fret'],
                        'string_name': p['string_name']})
            out.append(row)
    return sorted(out, key=lambda x: (x['start'], x['midi']))


# ── A/B: prox-viterbi (Cell 15) vs span-window (Cell 16) ─────────────────────
import io
from contextlib import redirect_stdout
def _render(a):
    b = io.StringIO()
    with redirect_stdout(b): render_ascii_tab(a)
    return b.getvalue()
def _mf(a):
    f=[n['fret'] for n in a if n.get('fret',0)>0]; return round(float(np.mean(f)),2) if f else 0.0
def _open(a): return sum(1 for n in a if n['fret']==0)

key = parse_key(KEY_LABEL)
in_range = [n for n in raw_notes if get_possible_positions(n['midi'])]
a_prox = assign_voiced_prox_viterbi(in_range, key=key)
a_span = assign_voiced_span_viterbi(in_range, key=key)

print(f"prox-viterbi : mean={_mf(a_prox):5}  open={_open(a_prox)}  max={max((n['fret'] for n in a_prox),default=0)}")
print(f"span-window  : mean={_mf(a_span):5}  open={_open(a_span)}  max={max((n['fret'] for n in a_span),default=0)}")

# Targeted check: where did each put the D4 (MIDI 62) notes?
print('\nD4 (MIDI 62) placements:')
for label, a in [('prox', a_prox), ('span', a_span)]:
    spots = [(round(n['start'],2), STRING_NAMES[n['string']], n['fret']) for n in a if n['midi']==62]
    print(f"  {label}: {spots}")

print('\n' + '='*64 + '\nPROX-VITERBI (Cell 15)\n' + '='*64)
print(_render(a_prox))
print('\n' + '='*64 + '\nSPAN-WINDOW (Cell 16)\n' + '='*64)
print(_render(a_span))

prox-viterbi : mean= 5.49  open=0  max=7
span-window  : mean= 5.49  open=0  max=7

D4 (MIDI 62) placements:
  prox: [(2.51, 'B', 3), (5.03, 'B', 3), (7.52, 'G', 7), (10.01, 'G', 7)]
  span: [(2.51, 'B', 3), (5.03, 'B', 3), (7.52, 'G', 7), (10.01, 'G', 7)]

PROX-VITERBI (Cell 15)
e|----------------5-------5-------|------------5-----------5-------|5-------------------5-------6---|6-------6-----------6-------6---|
B|------------6-------6-----------|5-------3-------6-----------6---|--------5---5---3-----------6---|----6-----------5---------------|
G|--------------------------------|--------------------------------|--------------------------------|--------3---------------7-------|
D|--------------------------------|--------------------------------|--------------------------------|--------------------------------|
A|--------------------------------|------------5-------------------|--------------------5-----------|--------------------------------|
E|--------------------------------|5---5-----

In [20]:
# ── Cell 17: register-filter THEN span-window decode ─────────────────────────
# Every prior assignment run used un-cleaned detection, so the hand-position
# estimate was poisoned by bass false positives. Filter first, then decode.

def drop_low_register_outliers(notes, window_sec=0.30, min_gap_semitones=9):
    if not notes: return notes
    starts = np.array([n['start'] for n in notes])
    midis  = np.array([n['midi']  for n in notes])
    keep = []
    for n in notes:
        near = np.abs(starts - n['start']) <= window_sec
        local = midis[near]
        ref = np.median(local[local >= np.median(local)])
        if n['midi'] <= ref - min_gap_semitones:
            continue
        keep.append(n)
    return keep

key = parse_key(KEY_LABEL)
filtered = drop_low_register_outliers(raw_notes)
in_range = [n for n in filtered if get_possible_positions(n['midi'])]
print(f"{len(raw_notes)} -> {len(filtered)} notes after register filter")

a_span_clean = assign_voiced_span_viterbi(in_range, key=key)

print(f"\nspan-window (filtered): mean={_mf(a_span_clean):5}  open={_open(a_span_clean)}  max={max((n['fret'] for n in a_span_clean),default=0)}")
print('\nD4 (MIDI 62) placements:')
spots = [(round(n['start'],2), STRING_NAMES[n['string']], n['fret']) for n in a_span_clean if n['midi']==62]
print(f"  {spots}")

print('\n' + '='*64 + '\nREGISTER-FILTER + SPAN-WINDOW\n' + '='*64)
print(_render(a_span_clean))

47 -> 37 notes after register filter

span-window (filtered): mean= 5.46  open=0  max=7

D4 (MIDI 62) placements:
  [(2.51, 'B', 3), (5.03, 'B', 3), (7.52, 'G', 7), (10.01, 'G', 7)]

REGISTER-FILTER + SPAN-WINDOW
e|----------------5-------5-------|------------5-----------5-------|5-------------------5-------6---|6-------6-----------6-------6---|
B|------------6-------6-----------|5-------3-------6-----------6---|--------5---5---3-----------6---|----6-----------5---------------|
G|--------------------------------|--------------------------------|--------------------------------|--------3---------------7-------|
D|--------------------------------|--------------------------------|--------------------------------|--------------------------------|
A|--------------------------------|--------------------------------|--------------------------------|--------------------------------|
E|--------------------------------|5---5---------------------------|--------------------------------|-----------

In [15]:
# ── Cell 14: onset_threshold sweep on this clip ──────────────────────────────
# Re-runs Basic Pitch at several onset_threshold values and shows how many
# low-register false-positive notes (the A2/A#2 bass blips) drop out, plus the
# resulting gated tab. Pure detection-precision probe — assignment layer unchanged.
import io
from contextlib import redirect_stdout

THRESHOLDS   = [0.50, 0.55, 0.60, 0.65]
BASS_CEILING = 55   # MIDI; notes below this (G3 and lower) are suspect for this melody

def _build_raw_notes(note_events):
    out = []
    for (s, e, p, amp, _) in note_events:
        if amp < BASIC_PITCH_AMPLITUDE_THRESHOLD:
            continue
        m = int(round(p))
        if not (BASIC_PITCH_MIN_MIDI <= m <= BASIC_PITCH_MAX_MIDI):
            continue
        out.append({'start': float(s), 'duration': float(e - s), 'midi': m,
                    'pitch_class': m % 12, 'amplitude': float(amp),
                    'in_key': None, 'in_chord': None, 'key_label': KEY_LABEL})
    out.sort(key=lambda n: n['start'])
    return out

def _render(notes):
    buf = io.StringIO()
    with redirect_stdout(buf):
        render_ascii_tab(notes)
    return buf.getvalue()

key = parse_key(KEY_LABEL)
rows = []
for th in THRESHOLDS:
    print(f'Running Basic Pitch @ onset_threshold={th} ...')
    _, _, ne = basic_pitch_predict(
        str(audio_path), onset_threshold=th, minimum_frequency=OPEN_STRING_MIDI[0])
    notes = _build_raw_notes(ne)
    n_total = len(notes)
    n_bass  = sum(1 for n in notes if n['midi'] < BASS_CEILING)
    in_range = [n for n in notes if get_possible_positions(n['midi'])]
    assigned = assign_caged_voiced_gated(in_range, key=key)
    rows.append({'onset_threshold': th, 'n_notes': n_total,
                 'n_bass_(<G3)': n_bass, 'n_melody': n_total - n_bass})
    print(f'  -> {n_total} notes  |  {n_bass} bass (<G3)  |  {n_total - n_bass} melody\n')

print('\nSummary — want bass count near 0 without losing melody notes:\n')
display(pd.DataFrame(rows))

# Print the tab at each threshold for visual comparison
for th in THRESHOLDS:
    _, _, ne = basic_pitch_predict(
        str(audio_path), onset_threshold=th, minimum_frequency=OPEN_STRING_MIDI[0])
    notes = [n for n in _build_raw_notes(ne) if get_possible_positions(n['midi'])]
    assigned = assign_caged_voiced_gated(notes, key=key)
    print('\n' + '=' * 64)
    print(f'GATED TAB @ onset_threshold = {th}')
    print('=' * 64)
    print(_render(assigned))

Running Basic Pitch @ onset_threshold=0.5 ...
Predicting MIDI for /content/drive/MyDrive/Capstone/Audio/isThisItIntro.mp3...
  -> 47 notes  |  13 bass (<G3)  |  34 melody

Running Basic Pitch @ onset_threshold=0.55 ...
Predicting MIDI for /content/drive/MyDrive/Capstone/Audio/isThisItIntro.mp3...
  -> 45 notes  |  12 bass (<G3)  |  33 melody

Running Basic Pitch @ onset_threshold=0.6 ...
Predicting MIDI for /content/drive/MyDrive/Capstone/Audio/isThisItIntro.mp3...
  -> 43 notes  |  11 bass (<G3)  |  32 melody

Running Basic Pitch @ onset_threshold=0.65 ...
Predicting MIDI for /content/drive/MyDrive/Capstone/Audio/isThisItIntro.mp3...
  -> 42 notes  |  10 bass (<G3)  |  32 melody


Summary — want bass count near 0 without losing melody notes:



,onset_threshold,n_notes,n_bass_(<G3),n_melody
0,0.50,47,13,34
1,0.55,45,12,33
2,0.60,43,11,32
3,0.65,42,10,32


Predicting MIDI for /content/drive/MyDrive/Capstone/Audio/isThisItIntro.mp3...

GATED TAB @ onset_threshold = 0.5
e|----------------5-------5-------|0-----------5-----------5-------|5-------0---0-------5-------6---|6-------6-------0---6-------6---|
B|------------6-------6-----------|--------3-------6-----------6---|----------------3-----------6---|----6---------------------------|
G|--------------------------------|--------------------------------|--------------------------------|------------------------7-------|
D|--------------------------------|--------------------------------|--------------------------------|--------8-----------------------|
A|--------------------------------|0---0-------5-------------------|--------0---0-------5-----------|--------------------------------|
E|--------------------------------|--------------------------------|--------------------------------|6---6---6-----------------------|

e|--------6-----------6---6---6---|----6---------------------------|
B|----

In [16]:
# ── Cell 15: register filter — drop low-register false positives ─────────────
# These A2/A#2 detections sit ~19 semitones below the F4/A4 melody they ride
# under. They're confident detections (so onset_threshold won't remove them),
# but they're sub-harmonic / sympathetic-string artifacts, not the part.

def drop_low_register_outliers(notes, window_sec=0.30, min_gap_semitones=9):
    """Drop a note if it sits >= min_gap_semitones BELOW the local melody median
    within +/- window_sec. Targets bass-register false positives under a melody."""
    if not notes:
        return notes
    starts = np.array([n['start'] for n in notes])
    midis  = np.array([n['midi']  for n in notes])
    keep = []
    for i, n in enumerate(notes):
        near = np.abs(starts - n['start']) <= window_sec
        local = midis[near]
        # local melody reference = median of the upper half (ignore the bass itself)
        ref = np.median(local[local >= np.median(local)])
        if n['midi'] <= ref - min_gap_semitones:
            continue  # drop: too far below the local melody
        keep.append(n)
    return keep

filtered = drop_low_register_outliers(raw_notes)
print(f'{len(raw_notes)} -> {len(filtered)} notes '
      f'({len(raw_notes) - len(filtered)} low-register outliers dropped)')

in_range = [n for n in filtered if get_possible_positions(n['midi'])]
assigned = assign_caged_voiced_gated(in_range, key=parse_key(KEY_LABEL))

print('\n' + '=' * 64)
print('GATED TAB — register-filtered')
print('=' * 64)
render_ascii_tab(assigned)

47 -> 37 notes (10 low-register outliers dropped)

GATED TAB — register-filtered
e|----------------5-------5-------|0-----------5-----------5-------|5-------0---0-------5-------6---|6-------6-------0---6-------6---|
B|------------6-------6-----------|--------3-------6-----------6---|----------------3-----------6---|----6---------------------------|
G|--------------------------------|--------------------------------|--------------------------------|------------------------7-------|
D|--------------------------------|--------------------------------|--------------------------------|--------8-----------------------|
A|--------------------------------|0---0---------------------------|--------------------------------|--------------------------------|
E|--------------------------------|--------------------------------|--------------------------------|--------------------------------|

e|--------6-----------6---6---6---|----6---------------------------|
B|----6-------6-----------5-------|----